In [4]:
!pip install keras-tuner==1.4.7
!pip install tensorflow==2.16.2 matplotlib==3.9.1 "numpy<2.0"
!pip install jax==0.4.29 jaxlib==0.4.29 ml_dtypes --upgrade

Reason for being yanked: The Windows wheels, under some conditions, caused segfaults in unrelated user code.  Due to this we deleted the Windows wheels to prevent these segfaults, however this caused greater disruption as pip then began to try (and fail) to build 3.9.1 from the sdist on Windows which impacted far more users.  Yanking the whole release is the only tool available to eliminate these failures without changes to on the user side.  The sdist, OSX wheel, and manylinux wheels are all functional and there are no critical bugs in the release.   Downstream packagers should not yank their builds of Matplotlib 3.9.1.  See https://github.com/matplotlib/matplotlib/issues/28551 for details.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.3/8.3 MB 35.6 MB/s eta 0:00:00
  Attempting uninstall: matplotlib
    Found existing installation: matplotlib 3.10.0
    Uninstalling matplotlib-3.10.0:
      Successfully uninstalled matplotlib-3.10.0


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 39.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.6/77.6 MB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.0/5.0 MB 121.8 MB/s eta 0:00:00
  Attempting uninstall: ml_dtypes
    Found existing installation: ml-dtypes 0.3.2
    Uninstalling ml-dtypes-0.3.2:
      Successfully uninstalled ml-dtypes-0.3.2
  Attempting uninstall: jaxlib
    Found existing installation: jaxlib 0.7.2
    Uninstalling jaxlib-0.7.2:
      Successfully uninstalled jaxlib-0.7.2
  Attempting uninstall: jax
    Found existing installation: jax 0.7.2
    Uninstalling jax-0.7.2:
      Successfully uninstalled jax-0.7.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tensorflow 2.16.2 requires ml-dtypes~=0.3.1, but you have ml-dtypes 0.5.4 which is incompatible.
tensorflow-text 2.20.1 requires tensorf

In [1]:
import sys

# Increase recursion limit to prevent potential issues
sys.setrecursionlimit(100000)

In [3]:
import keras_tuner as kt
from tensorflow.keras.datasets import mnist
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Flatten
from tensorflow.keras.optimizers import Adam
import os
import warnings

warnings.filterwarnings('ignore')

# Set TensorFlow log level to suppress warnings and info messages
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'  # 0 = all logs, 1 = filter out INFO, 2 = filter out INFO and WARNING, 3 = ERROR only


In [4]:
(x_train, y_train), (x_val, y_val) = mnist.load_data()
x_train, x_val = x_train/255., x_val/255.

print(f'Training data shape: {x_train.shape}')
print(f'Validation data shape: {x_val.shape}')

11490434/11490434 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Training data shape: (60000, 28, 28)
Validation data shape: (10000, 28, 28)


In [5]:
def build_model(hp):
  model = Sequential([
      Flatten(input_shape=(28, 28)),
      Dense(units=hp.Int('units', min_value=32, max_value=512, step=32), activation='relu'),
      Dense(10, activation='softmax')
  ])

  model.compile(
      optimizer=Adam(learning_rate=hp.Float('learning_rate', min_value=1e-4, max_value=1e-2, sampling='LOG')),
      loss='sparse_categorical_crossentropy',
      metrics=['accuracy']
  )

  return model

In [9]:
# Create a RandomSearch Tuner

tuner=kt.RandomSearch(
    build_model,
    objective='val_accuracy',
    max_trials=10,
    executions_per_trial=2,
    directory='random_search',
    project_name='mnist_classification'
)

# Display a summary of the search space
tuner.search_space_summary()

Search space summary
Default search space size: 2
units (Int)
{'default': None, 'conditions': [], 'min_value': 32, 'max_value': 512, 'step': 32, 'sampling': 'linear'}
learning_rate (Float)
{'default': 0.0001, 'conditions': [], 'min_value': 0.0001, 'max_value': 0.01, 'step': None, 'sampling': 'log'}


In [11]:
# Run the hyperparameter search
tuner.search(x_train, y_train, epochs=5, validation_data=(x_val, y_val))

# Display a summary of the results
tuner.results_summary()

Trial 10 Complete [00h 02m 11s]
val_accuracy: 0.9801999926567078

Best val_accuracy So Far: 0.980650007724762
Total elapsed time: 00h 17m 32s
Results summary
Results in random_search/mnist_classification
Showing 10 best trials
Objective(name="val_accuracy", direction="max")

Trial 02 summary
Hyperparameters:
units: 320
learning_rate: 0.000672349583281083
Score: 0.980650007724762

Trial 03 summary
Hyperparameters:
units: 288
learning_rate: 0.0006236776945025639
Score: 0.980650007724762

Trial 08 summary
Hyperparameters:
units: 448
learning_rate: 0.0009196308750140528
Score: 0.9805000126361847

Trial 09 summary
Hyperparameters:
units: 384
learning_rate: 0.0004600955422725711
Score: 0.9801999926567078

Trial 05 summary
Hyperparameters:
units: 512
learning_rate: 0.0006474873450435775
Score: 0.9796499907970428

Trial 06 summary
Hyperparameters:
units: 416
learning_rate: 0.0016478704855744067
Score: 0.9787499904632568

Trial 07 summary
Hyperparameters:
units: 192
learning_rate: 0.00239170668

In [13]:
# Step 1: Retrieve the best hyperparameters
best_hps = tuner.get_best_hyperparameters(num_trials=1)[0]

print('Best Hyperparameters:')
print(f'Units: {best_hps.get("units")}')
print(f'Learning Rate: {best_hps.get("learning_rate")}')

# Step 2: Build and Train the Model with Best Hyperparameters
model = tuner.hypermodel.build(best_hps)
model.fit(x_train, y_train, epochs=10, validation_data=(x_val, y_val))

# Evaluate the model on the test set
test_loss, test_acc = model.evaluate(x_val, y_val)
print(f'Test accuracy: {test_acc}')

Best Hyperparameters:
Units: 320
Learning Rate: 0.000672349583281083
Epoch 1/10
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 13s 6ms/step - accuracy: 0.9295 - loss: 0.2461 - val_accuracy: 0.9622 - val_loss: 0.1302
Epoch 2/10
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 19s 5ms/step - accuracy: 0.9706 - loss: 0.1010 - val_accuracy: 0.9759 - val_loss: 0.0820
Epoch 3/10
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 10s 5ms/step - accuracy: 0.9800 - loss: 0.0663 - val_accuracy: 0.9781 - val_loss: 0.0703
Epoch 4/10
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 10s 5ms/step - accuracy: 0.9857 - loss: 0.0462 - val_accuracy: 0.9760 - val_loss: 0.0761
Epoch 5/10
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 9s 5ms/step - accuracy: 0.9890 - loss: 0.0352 - val_accuracy: 0.9776 - val_loss: 0.0715
Epoch 6/10
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 11s 5ms/step - accuracy: 0.9923 - loss: 0.0258 - val_accuracy: 0.9772 - val_loss: 0.0714
Epoch 7/10
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 11s 6ms/step - accuracy: 0.9939 - loss: 0.0200 - val_accuracy: 0.9767 - val_loss: 0.0742
Epoch 8/10
1875